## 개인 예산 관리 AI Agent 프로젝트
Gemini Function Calling을 이용해 자연어로 개인 거래 내역과 예산을 관리하는 Agent를 구현한다. 다음 사용 예시를 처리하는 데 필요한 도구와 데이터 구조는 자유롭게 설계한다.

### 진행 방법
먼저 거래 등록, 검색, 수정과 예산 관리 기능을 Python 함수로 구현하고 직접 호출하여 동작을 확인한다.

각 함수가 정상적으로 동작하면 Gemini Function Calling에 도구로 연결하여 자연어 요청으로 같은 기능을 실행한다.

일단 해야할 것.
1. Gemini 연결
2. Function 생성

```
사용자: 오늘 점심으로 12,000원 썼어.
Agent: 지출 내역을 식비 카테고리로 등록한다.

사용자: 2026년 8월 27일 카페에서 쓴 금액을 4,500원으로 수정해줘.
Agent: 조건에 맞는 거래를 검색한 뒤 해당 내역을 수정한다.

사용자: 이번 달 식비 예산이 얼마나 남았어?
Agent: 식비 예산과 이번 달 지출을 조회하여 남은 금액을 알려준다.

사용자: 2026년 8월 지출 내역을 파일로 저장해줘.
Agent: 거래 내역을 조회한 뒤 보고서를 생성한다. (선택 기능)
```

### 핵심 기능
자연어로 수입과 지출 등록
날짜, 카테고리, 설명을 이용한 거래 검색
기존 거래 내역 수정
카테고리별 월 예산 설정
이번 달 사용 금액과 남은 예산 조회
잘못된 금액, 날짜, 거래 ID에 대한 예외 처리
Agent가 호출한 도구와 실행 결과를 콘솔에서 확인

### Agent 도구 설계
프로젝트의 핵심 기능을 Gemini가 호출할 수 있는 도구로 직접 설계한다.

도구는 다음 기능을 수행할 수 있어야 한다.

- [X] 수입과 지출 등록
- [X] 조건에 맞는 거래 검색 
- [X] 기존 거래 수정
- [X] 월별·카테고리별 예산 설정
- [X] 사용 금액과 남은 예산 조회
- [ ] 월별 거래 보고서 생성(선택)

### 데이터 구조
거래 내역에는 다음과 같은 정보를 저장할 수 있다.

- 거래 ID
- 거래 유형(수입 또는 지출)
- 카테고리
- 금액
- 설명
- 거래 날짜

Python 함수 구현
→ 함수 직접 호출 및 결과 확인
→ Gemini에 도구로 등록
→ 사용자 자연어 입력
→ Gemini가 필요한 도구 선택
→ Python 도구 함수 실행
→ 실행 결과를 Gemini에 반환
→ 추가 도구 호출 필요 여부 판단
→ 사용자에게 최종 결과 안내

### 1. import 설정 

In [43]:
# import 라이브러리
import json
import os
import requests

from pprint import pprint
from dotenv import load_dotenv
from google import genai
from datetime import date

from typing import Literal
from pydantic import BaseModel, ConfigDict, Field, ValidationError

In [44]:
# gemini 연결 준비
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError(".env 파일에 GEMINI_API_KEY를 설정해 주세요.")

model = os.getenv("GEMINI_MODEL", "gemini-3.6-flash")
client = genai.Client(api_key=api_key)
print("준비 완료 / 사용 모델:", model)

준비 완료 / 사용 모델: gemini-3.6-flash


### 2. 거래 내역 구조체 만들기


In [45]:
class Transaction(BaseModel):
    model_config = ConfigDict(extra="forbid")

    transaction_id: str = Field(
        description="거래를 구분하는 고유 ID. 예: T001"
    )
    transaction_type: Literal["수입", "지출"] = Field(
        description="거래 유형"
    )
    category: str = Field(
        description="거래 카테고리. 예: 식비, 교통, 급여"
    )
    amount: int = Field(
        ge=0,
        description="거래 금액. 0원 이상"
    )
    description: str = Field(
        max_length=100,
        description="거래에 관한 간단한 설명"
    )
    transaction_date: date = Field(
        description="거래 날짜. 예: 2026-08-28"
    )

# Pydantic 모델을 Gemini에 전달할 JSON Schema로 변환
# transaction_schema = Transaction.model_json_schema()

### 3. 함수

#### 3.1 내역 추가 함수

In [46]:
TRANSACTIONS = {}

In [128]:
def add_transaction(type, category, amount, description, date):

    if not isinstance(amount, int) or amount <= 0:
        return {
            "ok": False,
            "error": "금액은 0보다 큰 정수여야 합니다.",
        }

    # 날짜 검사 형식 준수
    try:
        datetime.strptime(date, "%Y-%m-%d")
    except ValueError:
        return {
            "ok": False,
            "error": "날짜는 YYYY-MM-DD 형식이어야 합니다.",
        }
        
    id = f"T{len(TRANSACTIONS) + 1:03}"

    TRANSACTIONS[id] = Transaction(
        transaction_id=id,
        transaction_type=type,
        category=category,
        amount=amount,
        description=description,
        transaction_date=date,
    )

    save_data()

    return {"ok": True,"message": "거래가 저장되었습니다.","transaction": TRANSACTIONS[id].model_dump(mode="json"),}

In [55]:
add_transaction_tool = {
    "type": "function",
    "name": "add_transaction",
    "description": "새 거래 내역을 저장합니다.",
    "parameters": {
        "type": "object",
        "properties": {
            "type": {
                "type": "string",
                "description": "거래 유형. 수입 또는 지출",
                "enum": ["수입", "지출"],
            },
            "category": {
                "type": "string",
                "description": "거래 카테고리. 예: 식비, 교통, 급여",
            },
            "amount": {
                "type": "integer",
                "description": "거래 금액. 0원 이상",
                "minimum": 0,
            },
            "description": {
                "type": "string",
                "description": "거래 설명. 예: 점심 식사",
            },
            "date": {
                "type": "string",
                "description": "거래 날짜. YYYY-MM-DD 형식. 예: 2026-08-28",
            },
        },
        "required": [
            "type",
            "category",
            "amount",
            "description",
            "date",
        ],
    },
}

#### 3.2 조건에 맞는 거래 검색

In [127]:
def search_transactions(date=None, category=None, description=None):
    # 조건에 맞는 구조체 저장
    results = []

    for transaction in TRANSACTIONS.values():
        if date and str(transaction.transaction_date) != date:
            continue

        if category and category not in transaction.category:
            continue

        if description and description.lower() not in transaction.description.lower():
            continue
        
        # model_dump(mode="json")는 pydantic을 json으로 변환
        results.append(transaction.model_dump(mode="json"))

    return {"ok": True, "count": len(results),"transactions": results,}

In [57]:
search_transactions_tool = {
    "type": "function",
    "name": "search_transactions",
    "description": "날짜, 카테고리 혹은 설명을 받아서 저장된 거래 내역을 찾는다.",
    "parameters": {
        "type": "object",
        "properties": {
            "date": {
                "type": "string",
                "description": "검색할 거래 날짜. ex) 2026-08-28",
            },
            "category": {
                "type": "string",
                "description": "검색할 카테고리. ex) 식비",
            },
            "description": {
                "type": "string",
                "description": "거래 설명에 포함된 단어. ex) 점심",
            },
        },
        "required": [],
    },
}

In [58]:
# print(search_transaction(category = "식비"))

#### 3.3 거래 수정


In [126]:
def update_transaction(id,type=None,category=None,amount=None,description=None,date=None,):
    transaction = TRANSACTIONS.get(id)

    if transaction is None:
        return {"ok": False,"error": "해당 거래 ID가 없습니다.",}

    data = transaction.model_dump()

    if type is not None:
        data["transaction_type"] = type

    if category is not None:
        data["category"] = category

    if amount is not None:
        data["amount"] = amount

    if description is not None:
        data["description"] = description

    if date is not None:
        data["transaction_date"] = date

    TRANSACTIONS[id] = Transaction(**data)

    save_data()

    return {"ok": True, "message": "거래가 수정되었습니다.","transaction": TRANSACTIONS[id].model_dump(mode="json"),}

In [73]:
update_transaction_tool = {
    "type": "function",
    "name": "update_transaction",
    "description": "기존 거래 내역을 수정합니다.",
    "parameters": {
        "type": "object",
        "properties": {
            "id": {
                "type": "string",
                "description": "수정할 거래 ID",
            },
            "type": {
                "type": "string",
                "enum": ["수입", "지출"],
            },
            "category": {
                "type": "string",
            },
            "amount": {
                "type": "integer",
                "minimum": 0,
            },
            "description": {
                "type": "string",
            },
            "date": {
                "type": "string",
                "description": "YYYY-MM-DD 형식",
            },
        },
        "required": ["id"],
    },
}

#### 3.4 월별·카테고리별 예산 설정


In [99]:
BUDGETS = {}

def set_budget(month, category, amount):
    if month not in BUDGETS:
        BUDGETS[month] = {}

    BUDGETS[month][category] = amount

    return {"ok": True,"month": month,"category": category,"amount": amount,}

In [100]:
set_budget_tool = {
    "type": "function",
    "name": "set_budget",  # 여기 수정
    "description": "특정 월과 카테고리의 예산을 설정하거나 수정합니다.",
    "parameters": {
        "type": "object",
        "properties": {
            "month": {
                "type": "string",
                "description": "예산을 설정할 연월. 예: 2026-08",
            },
            "category": {
                "type": "string",
                "description": "예산 카테고리. 예: 식비",
            },
            "amount": {
                "type": "integer",
                "description": "설정할 예산 금액",
                "minimum": 0,
            },
        },
        "required": ["month", "category", "amount"],
    },
}

#### 3.5 사용 금액과 남은 예산 조회

In [92]:
def check_budget(month, category):
    budget = BUDGETS.get(month, {}).get(category)

    if budget is None:
        return {
            "ok": False,
            "error": "설정된 예산이 없습니다.",
        }

    spent = 0

    for transaction in TRANSACTIONS.values():
        is_same_month = str(transaction.transaction_date).startswith(month)
        is_same_category = transaction.category == category
        is_expense = transaction.transaction_type == "지출"

        if is_same_month and is_same_category and is_expense:
            spent += transaction.amount

    return {
        "ok": True,
        "month": month,
        "category": category,
        "budget": budget,
        "spent": spent,
        "remaining": budget - spent,
    }

In [82]:
check_budget_tool = {
    "type": "function",
    "name": "check_budget",
    "description": "특정 월과 카테고리의 예산, 실제 지출, 남은 예산을 조회합니다.",
    "parameters": {
        "type": "object",
        "properties": {
            "month": {
                "type": "string",
                "description": "조회할 연월. YYYY-MM 형식. 예: 2026-08",
            },
            "category": {
                "type": "string",
                "description": "조회할 예산 카테고리. 예: 식비, 교통",
            },
        },
        "required": ["month", "category"],
    },
}

#### 3.6 문서 작성 함수


문서 작성은 지피티에게 자문을 구하여 구현 진행

In [118]:
from pathlib import Path
from docx import Document


def export_monthly_doc(month):
    doc = Document()
    doc.add_heading(f"{month} 거래 내역", level=1)

    count = 0

    for transaction in TRANSACTIONS.values():
        if str(transaction.transaction_date).startswith(month):
            text = (
                f"{transaction.transaction_date} | "
                f"{transaction.transaction_type} | "
                f"{transaction.category} | "
                f"{transaction.amount:,}원 | "
                f"{transaction.description}"
            )

            doc.add_paragraph(text)
            count += 1

    if count == 0:
        return {
            "ok": False,
            "error": f"{month} 거래 내역이 없습니다.",
        }

    doc.add_heading("카테고리별 예산 현황", level=2)

    for category, budget in BUDGETS.get(month, {}).items():
        spent = 0

        for transaction in TRANSACTIONS.values():
            if (
                str(transaction.transaction_date).startswith(month)
                and transaction.transaction_type == "지출"
                and transaction.category == category
            ):
                spent += transaction.amount

        doc.add_paragraph(
            f"{category} | "
            f"예산: {budget:,}원 | "
            f"지출: {spent:,}원 | "
            f"남은 금액: {budget - spent:,}원"
        )

    output_dir = Path("reports")
    output_dir.mkdir(exist_ok=True)

    output_path = output_dir / f"{month}_거래내역.docx"
    doc.save(output_path)

    return {
        "ok": True,
        "message": "문서가 저장되었습니다.",
        "file_path": str(output_path.resolve()),
    }

In [105]:
export_monthly_doc_tool = {
    "type": "function",
    "name": "export_monthly_doc",
    "description": "특정 월의 거래 내역을 Word 문서 파일로 저장합니다.",
    "parameters": {
        "type": "object",
        "properties": {
            "month": {
                "type": "string",
                "description": "저장할 연월. 예: 2026-08",
            },
        },
        "required": ["month"],
    },
}

#### 3.7 Json save 함수

In [125]:
DATA_PATH = Path("data/transactions_2026_04_to_08.json")

def save_data():
    data = {
        "budgets": BUDGETS,
        "transactions": [
            transaction.model_dump(mode="json")
            for transaction in TRANSACTIONS.values()
        ],
    }

    with open(DATA_PATH, "w", encoding="utf-8") as file:
        json.dump(data, file, ensure_ascii=False, indent=2)

### 메인 함수

In [107]:
TOOLS = [
    add_transaction_tool,
    search_transactions_tool,
    update_transaction_tool,
    set_budget_tool,
    check_budget_tool,
    export_monthly_doc_tool,
]

TOOL_FUNCTIONS = {
    "add_transaction": add_transaction,
    "search_transactions": search_transactions,
    "update_transaction": update_transaction,
    "set_budget" : set_budget,
    "check_budget" : check_budget,
    "export_monthly_doc" : export_monthly_doc,
}

4월 ~ 8일 거래 내역 json 파일을 불러온다. 

In [123]:
from pathlib import Path
import json

data_path = Path("data/transactions_2026_04_to_08.json")

with open(data_path, "r", encoding="utf-8") as file:
    data = json.load(file)

TRANSACTIONS = {
    item["transaction_id"]: Transaction(**item)
    for item in data["transactions"]
}

BUDGETS = data["budgets"]

print(f"거래 {len(TRANSACTIONS)}건을 불러왔습니다.")
print(BUDGETS["2026-08"])

거래 76건을 불러왔습니다.
{'식비': 300000, '교통': 100000}


In [95]:
from datetime import date as Date


In [112]:
SYSTEM_INSTRUCTION = f"""
너는 내 가계부 도와주는 에이전트야.
오늘은 {Date.today().isoformat()}이야.

- 수입이나 지출 얘기 나오면 add_transaction으로 바로 저장해.
- 날짜 없으면 오늘 날짜로 넣고, 밥·커피·마트 같은 건 알아서 적당히 분류해.
- 거래를 찾거나 보여 달라면 search_transactions를 써.
- 수정 요청인데 거래 ID가 없으면 먼저 검색해.
  하나만 나오면 update_transaction으로 수정하고,
  여러 개면 어떤 거래인지 사용자한테 물어봐.
- 도구 결과를 받기 전에는 저장·검색·수정됐다고 말하면 안 돼.
"""

In [115]:
def execute_tool_call(step) -> dict:
    tool_function = TOOL_FUNCTIONS.get(step.name)
    if tool_function is None:
        return {"ok": False, "error": f"허용되지 않은 도구: {step.name}"}

    try:
        return tool_function(**step.arguments)
    except TypeError as error:
        return {"ok": False, "error": f"잘못된 인자: {error}"}
    except Exception as error:
        return {"ok": False, "error": f"도구 실행 실패: {type(error).__name__}"}

def run_agent(user_input: str, max_turns: int = 5) -> dict:
    next_input = user_input
    previous_interaction_id = None
    logs = []

    for turn in range(1, max_turns + 1):
        request = {
            "model": model,
            "input": next_input,
            "tools": TOOLS,
            "system_instruction": SYSTEM_INSTRUCTION,
            "store": True,
        }
        if previous_interaction_id is not None:
            request["previous_interaction_id"] = previous_interaction_id

        interaction = client.interactions.create(**request)
        function_calls = [step for step in interaction.steps if step.type == "function_call"]

        if not function_calls:
            return {
                "ok": True,
                "answer": interaction.output_text,
                "turns": turn,
                "tool_logs": logs,
            }

        next_input = []
        for step in function_calls:
            result = execute_tool_call(step)
            logs.append({"turn": turn, "tool": step.name, "arguments": step.arguments, "result": result})
            next_input.append({
                "type": "function_result",
                "name": step.name,
                "call_id": step.id,
                "result": [{"type": "text", "text": json.dumps(result, ensure_ascii=False)}],
            })

        previous_interaction_id = interaction.id

    return {
        "ok": False,
        "answer": None,
        "turns": max_turns,
        "tool_logs": logs,
        "error": "최대 반복 횟수를 초과했습니다.",
    }

In [120]:
result = run_agent("8월 지출과 예산 보고서로 저장해줘")
print(result["answer"] if result["ok"] else result["error"])
print("\n도구 실행 기록")
pprint(result["tool_logs"])

2026년 8월 지출 및 거래 내역 보고서가 성공적으로 저장되었습니다.

* **저장 경로:** `C:\Users\김나린\Desktop\my-ai-agent\python\gemini\reports\2026-08_거래내역.docx`

도구 실행 기록
[{'arguments': {'month': '2026-08'},
  'result': {'file_path': 'C:\\Users\\김나린\\Desktop\\my-ai-agent\\python\\gemini\\reports\\2026-08_거래내역.docx',
             'message': '문서가 저장되었습니다.',
             'ok': True},
  'tool': 'export_monthly_doc',
  'turn': 1}]


In [97]:
result = run_agent("2026-08-25에 월급 300만원 들어왔어.")
print(result["answer"] if result["ok"] else result["error"])
print("\n도구 실행 기록")
pprint(result["tool_logs"])

2026년 8월 25일 자로 급여(수입) 3,000,000원(월급)이 정상적으로 저장되었습니다.

도구 실행 기록
[{'arguments': {'amount': 3000000,
                'category': '급여',
                'date': '2026-08-25',
                'description': '월급',
                'type': '수입'},
  'result': {'message': '거래가 저장되었습니다.',
             'ok': True,
             'transaction': {'amount': 3000000,
                             'category': '급여',
                             'description': '월급',
                             'transaction_date': '2026-08-25',
                             'transaction_id': 'T003',
                             'transaction_type': '수입'}},
  'tool': 'add_transaction',
  'turn': 1}]


In [131]:
# 7. 식비 예산 설정
result = run_agent("2026년 8월 식비 예산이 얼마야?.")
print(result["answer"] if result["ok"] else result["error"])
print("\n도구 실행 기록")
pprint(result["tool_logs"])

2026년 8월 식비 예산은 **500,000원**입니다.

- **예산**: 500,000원
- **현재 지출**: 109,000원
- **남은 예산**: 391,000원

도구 실행 기록
[{'arguments': {'category': '식비', 'month': '2026-08'},
  'result': {'budget': 500000,
             'category': '식비',
             'month': '2026-08',
             'ok': True,
             'remaining': 391000,
             'spent': 109000},
  'tool': 'check_budget',
  'turn': 1}]


In [102]:
# 8. 예산 확인
result = run_agent("2026년 8월 식비 예산이 얼마나 남았는지 알려줘.")
print(result["answer"] if result["ok"] else result["error"])
print("\n도구 실행 기록")
pprint(result["tool_logs"])

2026년 8월 식비 예산 현황입니다.

- **예산**: 300,000원
- **현재 지출**: 42,000원
- **남은 예산**: **258,000원**

도구 실행 기록
[{'arguments': {'category': '식비', 'month': '2026-08'},
  'result': {'budget': 300000,
             'category': '식비',
             'month': '2026-08',
             'ok': True,
             'remaining': 258000,
             'spent': 42000},
  'tool': 'check_budget',
  'turn': 1}]
